# Faza 4: Gručenje in klasifikacijsko modeliranje

**Zastavljen cilj**: Razdeliti podatkovno populacijo v smiselne profile s pomočjo nenadzorovanega učenja ter v nadaljevanju uspešno predvideti, kdo izmed njih predstavlja resnično tveganje za default (1). Slednje smo dosegli z implementacijo naprednega preverjanja metrik treh glavnih algoritmov: Logistične regresije, Naključnega gozda ter XGBoost algoritma.

Dokument predstavlja končno poročilo. Vse implementacije (todo-ji) mora razrešiti ekipa.

## 1. Priprava podatkov

Naložili smo finalizirano, končno bazo (iz katere smo v 3. fazi že odstranili čiste tekstualne značilke in dodali TF-IDF/NLP točke). Množica je sedaj popolnoma numerična in ne vsebuje nobenih praznih vrednosti.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# TODO (ekipa): Naložite podatke iz `../data/train_final.csv` v `train` in iz `../data/test_final.csv` v `test`.
# TODO (ekipa): Ločite značilke (X) in ciljno spremenljivko (y = `loan_status`) tako za učno kot testno množico.

## 2. Segmentacija - K-Means gručenje

Z namenom boljšega predvidevanja in razumevanja strank smo želeli uvesti dodatno agregacijsko sprememljivko - `risk_cluster`. Uporabili smo model K-Means. Optimalni **K** (število grozdov) smo prepoznali z matematično-grafično analizo (metoda komolca oz. Elbow Method).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# TODO (ekipa): Inicializirajte prazna seznama `wcss` in `sil_scores`. Z for zanko (npr. med k=2 do k=10) trenirajte `KMeans(n_clusters=k)`.
# TODO (ekipa): Zapišite `.inertia_` vrednost v wcss in izračunajte `silhouette_score` ter ga dodajte v sil_scores.
# TODO (ekipa): Narišite graf WCSS odziva (Elbow) IN graf silhuetnih vrednosti za potrditev optimalnega K.
# TODO (ekipa): Najdeni najboljši K izkoristite v dejanskem modelu KMeans, ga zgradite (.fit) na `X_train` in z .predict() dodajte gručo nazaj kot `X_train['cluster']` in enako za test.

## 3. Modeliranje A: Logistična regresija (Osnovni Baseline Model)

Za interpretativno sidro smo postavili preprosto logistično regresijo. Ker imamo opravka z močno asimetričnim zastopstvom, smo algoritem naučili na obravnavo nesimetrije tako, da smo mu poudarili utež `class_weight='balanced'`. Ocenitev modela smo vršili preko klasifikacijskega poročila na testni množici.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# TODO (ekipa): Inicializirajte in implementirajte `LogisticRegression(class_weight='balanced', max_iter=2000)`.
# TODO (ekipa): Naučite model na učnih podatkih in generirajte `.predict` nad The testno množico.
# TODO (ekipa): Prikažite izračune - narisana matrika zmede (seaborn heatmap), in izpis classification report.

## 4. Modeliranje B: Random Forest Classifier

Nelienerano razmerstvenih težav se Logistična Regresija običajno ne more uspešno dotankiti. Zato smo v potek uvrstili učenje odločitvenih dreves Naključnega Gozda. 
Okrnjeni razred 1 je obvezoval vpeljavo ustrezne porazdelitve, učenje določene max globine in števila stabilnih paralelnih dreves (`n_estimators`).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO (ekipa): Implementirajte Random Forest model. Uporabite `class_weight='balanced_subsample'`.
# TODO (ekipa): Za hitrost procesiranja uporabite RandomSearch, oz. ročno fiksirajte `max_depth` okoli 10-15.
# TODO (ekipa): Nauči in napovedi test! Izpiši matriko zmede ter klasifikacijsko poročilo za ta model.

## 5. Modeliranje C: XGBoost (Glavni klasifikator)

Dokončno težiščé modeliranja predstavlja gradientni prenos z algoritmi XGBoosting-a, ki ponuja redno pot iz nesimetričnih pasti z natančno utežitvijo gradienta napake vsake rešitve in prilagoditvijo `scale_pos_weight` atributa.

In [ ]:
import xgboost as xgb

# TODO (ekipa): Število vrstic izračunajte kot: `scale_weight = vrstic_z_status_0 / vrstic_z_status_1` neposredno v vaši y_train variabli.
# TODO (ekipa): Inicializirajte in zaženite `xgb.XGBClassifier(scale_pos_weight=scale_weight, random_state=42)`.
# TODO (ekipa): Prvoten rezultat učenja in fita izmerite s pomočjo konfuzijske matrike nad testno množico in obvezno izpišite metrike natančnosti/priklica/f1.

## 6. Primerjava in izvoz zmagovalnega modela

Za neposredno tehnično prepričanost smo celostne odzive vseh učečih entitet primerjali preko izrisa ROC krivulje. Najobsežnejši ROC pod-krivuljski faktor (AUC-ROC) nas je privedel do zmagovalnega modela, ki smo ga shranili kot neodvisno zunanje zmožno telo.

In [ ]:
import joblib

# TODO (ekipa): Z uporabo scikit metrike `roc_curve` na enem skupnem `plt.plot()` grafu vizualizirajte 3 ROC krivulje posamenih regresorskih napovedi `predict_proba`.
# TODO (ekipa): Zapišite jasno legendo z pripadajočimi AUC formati (LogisticRegression AUC: XX.XX itd.).
# TODO (ekipa): Shranite NAJBOLJŠI izbran model v mapo projektov -> `joblib.dump(zmagovalni_model, '../models/xgb_model.pkl')`.